# Wind Curl Comparison between HRDPS and CaSR

In this notebook, we compare the spatial and temporal difference of wind speed and wind (stress) curl between HRDPS and CaSR. The wind velocity and the curl affect the wind-driven mixing and upwelling.

## Basic Preparation



In [5]:
# path

path_casr_u='/ocean/jqiu/Atmospheric_RDPS/2008_2024_Integrated/Integrated_RDPS_P_UUC_10m_2008_2024.nc'

path_casr_u='/ocean/jqiu/Atmospheric_RDPS/2008_2024_Integrated/Integrated_RDPS_P_VVC_10m_2008_2024.nc'

import sys

sys.path.append("/home/jqiu/analysis-junqi/Tools-Junqi")

import junqi_nctool as jqnc

# an example of hrdps path
path_hrdps_u=jqnc.nc_path('wind_u',time='2018-01-01',resolution='hourly')


In [6]:
# display variables

jqnc.nc_disp(path_casr_u)

### NetCDF Summary

*Path: : `/ocean/jqiu/Atmospheric_RDPS/2008_2024_Integrated/Integrated_RDPS_P_VVC_10m_2008_2024.nc`*

#### Dimensions

Dimension,Size
time,149040
rlat,105
rlon,70


#### Variables

Name,Shape,Dimensions
rotated_pole,(),scalar
CaSR_v3.2_P_VVC_10m,"(149040, 105, 70)","time, rlat, rlon"
time,"(149040,)",time
lon,"(105, 70)","rlat, rlon"
lat,"(105, 70)","rlat, rlon"
rlon,"(70,)",rlon
rlat,"(105,)",rlat


In [7]:
# display variables

jqnc.nc_disp(path_hrdps_u)

### NetCDF Summary

*Path: : `/results/forcing/atmospheric/GEM2.5/operational/ops_y2018m01d01.nc`*

#### Dimensions

Dimension,Size
time_counter,24
y,266
x,256


#### Variables

Name,Shape,Dimensions
atmpres,"(24, 266, 256)","time_counter, y, x"
nav_lat,"(266, 256)","y, x"
nav_lon,"(266, 256)","y, x"
percentcloud,"(24, 266, 256)","time_counter, y, x"
precip,"(24, 266, 256)","time_counter, y, x"
qair,"(24, 266, 256)","time_counter, y, x"
solar,"(24, 266, 256)","time_counter, y, x"
tair,"(24, 266, 256)","time_counter, y, x"
therm_rad,"(24, 266, 256)","time_counter, y, x"
u_wind,"(24, 266, 256)","time_counter, y, x"


In [17]:
# combine hrdps dataset

import xarray as xr

def variables(ds):
    return ds[['u_wind','v_wind','nav_lat','nav_lon']]

ds_hrdps=xr.open_mfdataset("/results/forcing/atmospheric/GEM2.5/operational/ops_y2018*.nc", preprocess=variables,combine='by_coords',chunks={'time_counter': 24})

/tmp/ipykernel_1613116/2711121416.py:8: FutureWarning: In a future version of xarray the default value for data_vars will change from data_vars='all' to data_vars=None. This is likely to lead to different results when multiple datasets have matching variables with overlapping values. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=True) or set data_vars explicitly.
  ds_hrdps=xr.open_mfdataset("/results/forcing/atmospheric/GEM2.5/operational/ops_y2018*.nc", preprocess=variables,combine='by_coords',chunks={'time_counter': 24})


In [18]:
# check combined hrdps dataset

print(ds_hrdps)

<xarray.Dataset> Size: 14GB
Dimensions:       (time_counter: 8760, y: 266, x: 256)
Coordinates:
  * time_counter  (time_counter) datetime64[ns] 70kB 2018-01-01 ... 2018-12-3...
  * x             (x) float64 2kB 0.0 2.5e+03 5e+03 ... 6.35e+05 6.375e+05
  * y             (y) float64 2kB 0.0 2.5e+03 5e+03 ... 6.6e+05 6.625e+05
Data variables:
    u_wind        (time_counter, y, x) float32 2GB dask.array<chunksize=(24, 266, 256), meta=np.ndarray>
    v_wind        (time_counter, y, x) float32 2GB dask.array<chunksize=(24, 266, 256), meta=np.ndarray>
    nav_lat       (time_counter, y, x) float64 5GB dask.array<chunksize=(24, 266, 256), meta=np.ndarray>
    nav_lon       (time_counter, y, x) float64 5GB dask.array<chunksize=(24, 266, 256), meta=np.ndarray>
Attributes:
    Conventions:          CF-1.0
    History:              Mon Jan  1 10:36:51 2018: ncks -4 -L4 -O /results/f...
    GRIB2_grid_template:  20
    NCO:                  4.4.2


In [ ]:
# calculate absolute velocity

import numpy as np

velocity_hrdps=np.sqrt(ds_hrdps['u_wind']**2+ds_hrdps['v_wind']**2)

# use meter to calculate du/dy and dv/dx

vorticity_hrdps=da_hrdps[u_wind].differentiate('y')-da_hrdps[v_wind].differentiate('x')

# calculate mean

mean_velocity_hrdps=velocity_hrdps.mean(dim='time_counter')
mean_vorticity_hrdps=vorticity_hrdps.mean(dim='time_counter')

mean_hrdps_2018=xr.Dataset({
    'mean_velocity': mean_velocity_hrdps,
    'mean_vorticity': mean_vorticity_hrdps
})

In [ ]:
# save

output_path = "hrdps_2018_mean_wind_and_curl.nc"

mean_hrdps_2018.to_netcdf(output_path, encoding=encoding)